PART D - Imbalance & Operational Stress
(imbalance = arrivals - departures )

For Part D, I load the trip data, count how many arrivals and departures each station has, and compute the imbalance using the formula:

imbalance = arrivals − departures.
Then I list the stations that keep emptying, the stations that keep overfilling, and the ones that change the most day-to-day.


In [1]:
import pandas as pd

files = [
    "202501-citibike-tripdata_1.csv",
    "202502-citibike-tripdata_1.csv",
    "202503-citibike-tripdata_1.csv",
    "202504-citibike-tripdata_4.csv"
]

dfs = []
for f in files:
    temp = pd.read_csv(f)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")

imbalance = pd.concat([arrivals, departures], axis=1).fillna(0)
imbalance["imbalance"] = imbalance["arrivals"] - imbalance["departures"]

top_deficits = imbalance.sort_values("imbalance").head(15)
top_surplus = imbalance.sort_values("imbalance", ascending=False).head(15)

df["date"] = df["started_at"].dt.date
daily_counts = df.groupby(["date", "start_station_name"]).size().unstack(fill_value=0)
volatility = daily_counts.std().sort_values(ascending=False)

print("=== Stations That Keep Emptying (Deficits) ===")
print(top_deficits)

print("\n=== Stations That Keep Overfilling (Surplus) ===")
print(top_surplus)

print("\n=== Most Volatile Stations (Day-to-Day Changes) ===")
print(volatility.head(15))


/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/1761121275.py:12: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/1761121275.py:12: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/1761121275.py:12: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/1761121275.py:12: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)


=== Stations That Keep Emptying (Deficits) ===
                            arrivals  departures  imbalance
Broadway & E 14 St              7397     11630.0    -4233.0
6 Ave & W 33 St                 6497     10365.0    -3868.0
Pier 61 at Chelsea Piers        9806     13436.0    -3630.0
7 Ave & Central Park South      5065      8102.0    -3037.0
11 Ave & W 27 St                3304      6334.0    -3030.0
W 54 St & 11 Ave                4052      6780.0    -2728.0
West St & Liberty St            4257      6813.0    -2556.0
1 Ave & E 68 St                 7589     10054.0    -2465.0
E 32 St & Park Ave              4938      7342.0    -2404.0
W 70 St & Amsterdam Ave         4181      6378.0    -2197.0
E 55 St & 2 Ave                 2393      4513.0    -2120.0
E 74 St & 1 Ave                 3442      5547.0    -2105.0
Broadway & W 61 St              3522      5621.0    -2099.0
W 67 St & Broadway              4206      6294.0    -2088.0
W 59 St & 10 Ave                3239      5291.0    -

PART F — Network Inference & Link Prediction

For Part F, I build a station network from the trip data and then run three link-prediction methods: Adamic–Adar, Preferential Attachment, and Common Neighbors. These scores help me find “emerging” connections between stations, meaning station pairs that might grow into strong corridors in the future even if they don’t currently have many trips.

In [2]:
import pandas as pd
import networkx as nx

files = [
    "202501-citibike-tripdata_1.csv",
    "202502-citibike-tripdata_1.csv",
    "202503-citibike-tripdata_1.csv",
    "202504-citibike-tripdata_4.csv"
]

dfs = []
for f in files:
    temp = pd.read_csv(f)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

G = nx.DiGraph()

for _, row in df.iterrows():
    u = row["start_station_name"]
    v = row["end_station_name"]
    if pd.isna(u) or pd.isna(v):
        continue
    if G.has_edge(u, v):
        G[u][v]["weight"] += 1
    else:
        G.add_edge(u, v, weight=1)

G_u = nx.Graph()

for u, v, data in G.edges(data=True):
    w = data["weight"]
    if G_u.has_edge(u, v):
        G_u[u][v]["weight"] += w
    else:
        G_u.add_edge(u, v, weight=w)

aa_scores = list(nx.adamic_adar_index(G_u))
aa_sorted = sorted(aa_scores, key=lambda x: x[2], reverse=True)
top_adamic_adar = aa_sorted[:20]

pa_scores = list(nx.preferential_attachment(G_u))
pa_sorted = sorted(pa_scores, key=lambda x: x[2], reverse=True)
top_preferential_attachment = pa_sorted[:20]

cn_scores = list(nx.common_neighbor_centrality(G_u))
cn_sorted = sorted(cn_scores, key=lambda x: x[2], reverse=True)
top_common_neighbors = cn_sorted[:20]

print("=== Top Adamic-Adar Predictions ===")
print(top_adamic_adar)

print("\n=== Top Preferential Attachment Predictions ===")
print(top_preferential_attachment)

print("\n=== Top Common Neighbor Predictions ===")
print(top_common_neighbors)


/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/593447268.py:13: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/593447268.py:13: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/593447268.py:13: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)
/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_3898/593447268.py:13: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f)


=== Top Adamic-Adar Predictions ===
[('Cleveland Pl & Spring St', '1 Ave & E 68 St', 91.02934569087063), ('Jay St & Tech Pl', 'S 5 Pl & S 5 St', 90.56816464270241), ('1 Ave & E 68 St', 'Greenwich Ave & 8 Ave', 89.17787213981306), ('Bergen St & Vanderbilt Ave', 'Scholes St & Manhattan Ave', 87.61531146694597), ('Jay St & Tech Pl', 'Driggs Ave & S 2 St', 86.58942014102703), ('S 5 Pl & S 5 St', 'Fulton St & Waverly Ave', 86.48037463060423), ('Lafayette Ave & Ft Greene Pl', 'Allen St & Stanton St', 85.32964524374496), ('S 3 St & Bedford Ave', 'Centre St & Chambers St', 85.15602156335878), ('Lawrence St & Willoughby St', 'Roebling St & N 4 St', 85.03885774756978), ('S 4 St & Roebling St', 'Willoughby Ave & Hall St', 84.55796736007632), ('Grand St & Havemeyer St', 'Fulton St & Adams St', 84.18008812149277), ('Bond St & Fulton St', 'Hope St & Union Ave', 83.25377098137372), ('Metropolitan Ave & Bedford Ave', 'St Marks Pl & 1 Ave', 82.31278868477999), ('Great Jones St', 'Fulton St & Adams St',